In [1]:
import os
os.environ['HF_HUB_DISABLE_XET'] = '1'


In [2]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'


In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
import hashlib
import os
import subprocess
import sys
from pathlib import Path

WORKDIR = Path(os.environ.get('SNU_WORKDIR', '/content/snu_ai'))
DRIVE_ROOT = Path(os.environ.get('SNU_DRIVE_ROOT', '/content/drive/MyDrive/snu ai'))
LOCAL_DATA_ROOT = WORKDIR / 'snuaichallenge_data'
DRIVE_DATA_ROOT = DRIVE_ROOT / 'snuaichallenge_data'

# scripts/build_pipelinec_bundle.py 출력값. 번들을 다시 만들면 갱신할 것.
EXPECTED_BUNDLE_SHA256 = '4FB9777B91F38C7E949001E27007FEEFD0216048F385373AFEFD0E72A760831A'

def has_train_data(root):
    return (root / 'train.csv').is_file() and (root / 'train').is_dir()

def find_drive_file(filename):
    direct = DRIVE_ROOT / filename
    if direct.is_file():
        return direct
    matches = list(Path('/content/drive/MyDrive').rglob(filename))
    return matches[0] if matches else None

WORKDIR.mkdir(parents=True, exist_ok=True)
if has_train_data(DRIVE_DATA_ROOT):
    DATA_ROOT = DRIVE_DATA_ROOT
else:
    DATA_ROOT = LOCAL_DATA_ROOT
    if not has_train_data(DATA_ROOT):
        train_bundle = find_drive_file('snu_colab_full_train_bundle.tar')
        if train_bundle is None:
            raise FileNotFoundError(
                "MyDrive에서 snu_colab_full_train_bundle.tar를 찾지 못했습니다. "
                "MyDrive/snu ai/에 업로드해 주세요."
            )
        print(f'학습 데이터 압축 해제: {train_bundle}')
        subprocess.run(['tar', '-xf', str(train_bundle), '-C', str(WORKDIR)], check=True)
        DRIVE_ROOT = train_bundle.parent
assert has_train_data(DATA_ROOT), f'학습 데이터가 없습니다: {DATA_ROOT}'

bundle_path = find_drive_file('snu_colab_pipelinec_bundle.tar')
if bundle_path is None:
    raise FileNotFoundError(
        "MyDrive에서 snu_colab_pipelinec_bundle.tar를 찾지 못했습니다. "
        "scripts/build_pipelinec_bundle.py로 만들어 MyDrive/snu ai/에 업로드해 주세요."
    )
bundle_sha = hashlib.sha256(bundle_path.read_bytes()).hexdigest().upper()
assert bundle_sha == EXPECTED_BUNDLE_SHA256, f'번들 sha256 불일치: {bundle_sha}'
subprocess.run(['tar', '-xf', str(bundle_path), '-C', str(WORKDIR)], check=True)

if str(WORKDIR) not in sys.path:
    sys.path.insert(0, str(WORKDIR))

DATA_DIR = str(DATA_ROOT)
IMAGE_ROOT = DATA_ROOT / 'train'
OUTPUT_DIR = DRIVE_ROOT / 'snu_pipelinec_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FOLD_ASSIGNMENTS_CSV = WORKDIR / 'outputs/lora_fold_split/fold_assignments.csv'
FOLD0_COSINE_CSV = WORKDIR / 'outputs/hybrid_judgment/frame_pairwise_cosine.csv'
TRAIN_COSINE_CSV = WORKDIR / 'outputs/qwen3vl_v2_os_selector/train_pairwise_cosine.csv'
FT_VAL_PREDS_CSV = WORKDIR / 'outputs/lora_holdout_infer/ft_val_preds.csv'
FOLD0_FEATURES_PT = WORKDIR / 'outputs/hybrid_judgment/image_features.pt'
for path in (FOLD_ASSIGNMENTS_CSV, FOLD0_COSINE_CSV, TRAIN_COSINE_CSV,
             FT_VAL_PREDS_CSV, FOLD0_FEATURES_PT):
    assert path.is_file(), f'번들에서 풀리지 않은 파일: {path}'

print(f'DATA_DIR: {DATA_DIR}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')


학습 데이터 압축 해제: /content/drive/MyDrive/snu ai/snu_colab_full_train_bundle.tar
DATA_DIR: /content/snu_ai/snuaichallenge_data
OUTPUT_DIR: /content/drive/MyDrive/snu ai/snu_pipelinec_outputs


In [5]:
# qwen3vl_v2_os 노트북과 동일한 핀. 기준선 adapter의 holdout EM 0.5298을 재현한
# 조합이며, Colab 기본 torchao 0.10은 peft LoRA 주입을 깨뜨리므로 함께 올린다.
import subprocess
import sys

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet',
    'transformers==5.13.0',
    'accelerate==1.14.0',
    'bitsandbytes>=0.46',
    'qwen-vl-utils==0.0.14',
    'peft',
    'torchao>=0.16.0',
], check=True)
print('설치 완료')


설치 완료


In [6]:
import json
import random
import time
from pathlib import Path

import numpy as np
import torch

# ---- 실험 상수 (Phase 0 보고 후 확정) ----------------------------------
# 이번엔 tau 3종 각각에 별도 STAGE 3 어댑터를 학습한다(각 tau가 자기 그룹에 매칭돼
# 외삽이 없다). test.csv도 tau별로 만든다. 대가: 학습 시간 3배(총 20~26h, 2~3 세션).
TRAIN_TAUS = (0.80, 0.85, 0.90)
EVAL_TAUS = TRAIN_TAUS              # 모든 tau가 자기 tau로 학습됨 - 외삽 없음
# STAGE 2 그룹 간 순서 팔. c1(캡션 앵커)은 EM 0.08~0.22로 무의미해 제거.
#   b   = FT 조대순서(ft_pred를 그룹 경계로 접음)
#   c2i = 그룹 대표 1장씩만 FT에 재질의
STAGE2_ARMS = ('b', 'c2i')

def tau_tag(tau):
    return f't{int(round(tau * 100)):03d}'   # 0.85 -> 't085'

# v2: STAGE 3 프롬프트 교체(near-dup 단일 순간 전용 + 방향성 단서 명시). 옛 프롬프트로
# 학습된 pipelinec_s3_* 어댑터와는 안 맞으므로 이름을 바꿔 깨끗이 재학습한다. 옛 체크포인트는
# Drive에 남지만 이 이름으로는 참조되지 않는다.
ARM_NAME = 'pipelinec_s3v2'
MODEL_NAME = 'Qwen/Qwen3-VL-8B-Instruct'
MODEL_REVISION = '0c351dd01ed87e9c1b53cbc748cba10e6187ff3b'
WARM_START_ADAPTER = DRIVE_ROOT / 'snu_qwen3vl_outputs' / 'qwen3vl_final'
SEED = 42
MAX_EPOCHS = 3                      # 계획상 상한. 늘리지 말 것.
# early stopping은 "언제 멈출지"만 정한다. 매 epoch 후 그룹 내 순서 정확도만
# 프록시로 재고(전체 조립 없음), best 대비 개선이 이 값 미만이면 중단.
# 채택 여부는 여기서 전혀 판단하지 않는다 - 그건 학습이 끝난 뒤 EM으로만 본다.
EARLY_STOP_MIN_DELTA = 0.005        # 0.5%p
GRAD_ACCUM_STEPS = 8                # 기존 레시피 동결
LEARNING_RATE = 2e-4
MAX_GRAD_NORM = 1.0
# 학습 약 13시간(4.4h/epoch) - Colab 세션이 끊길 수 있으므로 스텝 체크포인트 필수.
CHECKPOINT_EVERY_STEPS = 2000       # 약 1시간마다. 최악의 재작업량이 이만큼이다.
# parse 실패 시 fallback은 identity('제시 순서 그대로')인데, 그 fallback이 우연히
# 정답이 되는 비율이 33~43%다(No_ordering=True는 GT가 identity라 자동 정답).
# 채택 기준은 EM 하나뿐이므로 이 값은 게이트가 아니라 경고선이다 - 넘으면 EM이
# 실력이 아니라 fallback으로 부풀려졌을 수 있다는 뜻.
PARSE_FAILURE_WARN = 0.02
HOLDOUT_FOLD = 0
EXAMPLE_IDS = ('8zqOpJ', '8oZheL', 'W83k9f')
HOLDOUT_IDS_SHA256 = '82d69f13a10d3f1903694201d020c85d8ca7d71c1c07ae17d2ef80a0bd2f44d0'

GOLDEN = {
    "ft_em": 0.5298,
    "n_scored": 1914,
    "0.8": {
        "ceiling": 0.9357,
        "floor": 0.5167,
        "oracle": 0.8647
    },
    "0.85": {
        "ceiling": 0.9206,
        "floor": 0.5178,
        "oracle": 0.7821,
        "within": {
            "2": 0.769,
            "3": 0.469,
            "4": 0.307
        }
    },
    "0.9": {
        "ceiling": 0.9253,
        "floor": 0.5225,
        "oracle": 0.6599,
        "within": {
            "2": 0.716,
            "3": 0.39,
            "4": 0.216
        }
    }
}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

assert WARM_START_ADAPTER.is_dir(), (
    f'warm-start 어댑터가 없습니다: {WARM_START_ADAPTER}. '
    'qwen3vl_final은 Drive의 snu_qwen3vl_outputs/ 아래에 있어야 합니다.'
)

gpu_name = torch.cuda.get_device_name(0)
gpu_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
print(f'GPU: {gpu_name} ({gpu_vram_gb:.1f} GiB)')
assert gpu_vram_gb >= 38, (
    'STAGE 3 학습은 픽셀 캡 없는 LoRA 레시피라 A100-40GB급에서만 검증됐습니다.'
)
print(f'TRAIN_TAUS={TRAIN_TAUS}  ARM_NAME={ARM_NAME}  MAX_EPOCHS={MAX_EPOCHS}')


GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition (95.0 GiB)
TRAIN_TAUS=(0.8, 0.85, 0.9)  ARM_NAME=pipelinec_s3v2  MAX_EPOCHS=3


In [7]:
# ---- Phase 0: 자산 로드 + 골든값 배관 검증 ------------------------------
# 학습 전에 조립 로직이 맞는지 확인한다. 여기서 틀리면 GPU를 쓸 이유가 없다.
import ast
import collections
import csv
import hashlib

from snu_ordering.pipelinec import (
    build_full_prompt,
    build_group_prompt,
    build_representative_prompt,
    components,
    group_members,
    group_order_from_rank_label,
    groups_contiguous,
    multi_frame_groups,
    parse_group_output,
    reassemble,
    stage3_examples_for_sample,
    temporal_order_to_rank_label,
    within_group_order,
)

def read_csv_rows(path):
    with open(path, encoding='utf-8-sig', newline='') as handle:
        return list(csv.DictReader(handle))

def cosine_map(path):
    return {row['Id']: {k: float(v) for k, v in row.items() if k.startswith('cos_')}
            for row in read_csv_rows(path)}

fold_rows = read_csv_rows(FOLD_ASSIGNMENTS_CSV)
fold_by_id = {r['Id']: int(r['Fold']) for r in fold_rows}
answer_by_id = {r['Id']: ast.literal_eval(r['Answer']) for r in fold_rows}
no_ordering_by_id = {r['Id']: r['No_ordering'].strip().lower() == 'true' for r in fold_rows}
train_csv_by_id = {r['Id']: r for r in read_csv_rows(Path(DATA_DIR) / 'train.csv')}

holdout_ids = sorted(i for i, f in fold_by_id.items() if f == HOLDOUT_FOLD)
digest = hashlib.sha256('\n'.join(holdout_ids).encode('utf-8')).hexdigest()
assert digest == HOLDOUT_IDS_SHA256, f'holdout 지문 불일치: {digest}'
assert len(holdout_ids) == 1915, f'fold-0 행 수 이상: {len(holdout_ids)}'

ft_pred = {r['Id']: ast.literal_eval(r['pred']) for r in read_csv_rows(FT_VAL_PREDS_CSV)}
fold0_cos = cosine_map(FOLD0_COSINE_CSV)
train_cos = cosine_map(TRAIN_COSINE_CSV)

# 학습셋에 fold-0이 새면 판정 전체가 무효가 된다.
leak = set(train_cos) & set(holdout_ids)
assert not leak, f'train cosine에 fold-0 id가 섞였습니다: {sorted(leak)[:5]}'
assert len(train_cos) == 7620, f'train fold 행 수 이상: {len(train_cos)}'

scored_ids = [i for i in holdout_ids if i not in EXAMPLE_IDS]
assert len(scored_ids) == GOLDEN['n_scored']

def assemble_with_stub(sample_id, tau, stage3):
    """STAGE 3 자리에 ft/oracle을 꽂은 조립 - 파이프라인의 바닥과 천장."""
    comp = components(fold0_cos[sample_id], tau)
    order = group_order_from_rank_label(ft_pred[sample_id], comp)
    source = answer_by_id[sample_id] if stage3 == 'oracle' else ft_pred[sample_id]
    within = {g: within_group_order(source, m) for g, m in group_members(comp).items()}
    return reassemble(order, within, comp)

def rate(predicate, ids):
    return sum(bool(predicate(i)) for i in ids) / len(ids)

ft_em = rate(lambda i: answer_by_id[i] == ft_pred[i], scored_ids)
assert round(ft_em, 4) == GOLDEN['ft_em'], f'FT baseline 재현 실패: {ft_em:.4f}'
print(f'FT baseline EM = {ft_em:.4f}  (n={len(scored_ids)})')

for tau_key, expected in GOLDEN.items():
    if not isinstance(expected, dict):
        continue
    tau = float(tau_key)
    measured = (
        round(rate(lambda i: groups_contiguous(answer_by_id[i], components(fold0_cos[i], tau)), scored_ids), 4),
        round(rate(lambda i: assemble_with_stub(i, tau, 'ft') == answer_by_id[i], scored_ids), 4),
        round(rate(lambda i: assemble_with_stub(i, tau, 'oracle') == answer_by_id[i], scored_ids), 4),
    )
    golden = (expected['ceiling'], expected['floor'], expected['oracle'])
    assert measured == golden, f'tau={tau} 골든값 불일치: {measured} != {golden}'
    print(f'  tau={tau:.2f}  ceiling={measured[0]}  floor={measured[1]}  oracle={measured[2]}  OK')

print('골든값 배관 검증 통과 - 조립 로직 정상')


FT baseline EM = 0.5298  (n=1914)
  tau=0.80  ceiling=0.9357  floor=0.5167  oracle=0.8647  OK
  tau=0.85  ceiling=0.9206  floor=0.5178  oracle=0.7821  OK
  tau=0.90  ceiling=0.9253  floor=0.5225  oracle=0.6599  OK
골든값 배관 검증 통과 - 조립 로직 정상


In [8]:
# ---- Phase 0 산출물: component_dist.md ---------------------------------
def shape_counts(cos_map, ids, tau):
    counter = collections.Counter()
    for sample_id in ids:
        sizes = collections.Counter(components(cos_map[sample_id], tau)).values()
        counter[tuple(sorted(sizes, reverse=True))] += 1
    return counter

def dist_rows(cos_map, ids, label):
    lines = []
    for tau in (0.80, 0.85, 0.90):
        counts = shape_counts(cos_map, ids, tau)
        n = len(ids)
        by_max = collections.Counter()
        for shape, count in counts.items():
            by_max[max(shape)] += count
        cells = ' / '.join(f'{by_max[k] / n * 100:.1f}' for k in (1, 2, 3, 4))
        multi = (n - by_max[1]) / n * 100
        lines.append(f'| {label} | {tau:.2f} | {n} | {cells} | {multi:.1f}% |')
    return lines

report = ['# Pipeline C - Phase 0: component 크기 분포', '',
          f'- CLIP: openai/clip-vit-large-patch14 (기존 산출 재사용, 재계산 없음)',
          f'- fold-0 = eval (n={len(scored_ids)} scored), train = fold 1~4 (n={len(train_cos)})',
          '', '| set | tau | n | size 1/2/3/4 (%) | size>=2 |', '|---|---|---|---|---|']
report += dist_rows(train_cos, sorted(train_cos), 'TRAIN f1-4')
report += dist_rows(fold0_cos, scored_ids, 'FOLD0')
report += dist_rows(fold0_cos, [i for i in scored_ids if not no_ordering_by_id[i]],
                    'FOLD0 No_ordering=F')

# STAGE 3가 볼 수 있는 FT 오답 비율 - tau 하한 근거
report += ['', '## tau별 FT 오답 커버리지', '',
           '| tau | FT 오답 중 GT-인접 max cos > tau (STAGE 3가 볼 수 있음) |', '|---|---|']
wrong = [i for i in scored_ids if answer_by_id[i] != ft_pred[i]]
for tau in (0.80, 0.85, 0.90):
    covered = 0
    for sample_id in wrong:
        order = [im for im in sorted(range(1, 5), key=lambda x: answer_by_id[sample_id][x - 1])]
        adj = [fold0_cos[sample_id][f'cos_{min(a, b)}{max(a, b)}']
               for a, b in zip(order, order[1:])]
        covered += max(adj) > tau
    report.append(f'| {tau:.2f} | {covered / len(wrong) * 100:.1f}% |')

COMPONENT_DIST_MD = OUTPUT_DIR / 'component_dist.md'
COMPONENT_DIST_MD.write_text('\n'.join(report) + '\n', encoding='utf-8')
print('\n'.join(report))
print(f'\n-> {COMPONENT_DIST_MD}')


# Pipeline C - Phase 0: component 크기 분포

- CLIP: openai/clip-vit-large-patch14 (기존 산출 재사용, 재계산 없음)
- fold-0 = eval (n=1914 scored), train = fold 1~4 (n=7620)

| set | tau | n | size 1/2/3/4 (%) | size>=2 |
|---|---|---|---|---|
| TRAIN f1-4 | 0.80 | 7620 | 10.9 / 36.5 / 15.5 / 37.1 | 89.1% |
| TRAIN f1-4 | 0.85 | 7620 | 25.3 / 39.3 / 14.1 / 21.3 | 74.7% |
| TRAIN f1-4 | 0.90 | 7620 | 53.4 / 30.8 / 8.5 / 7.3 | 46.6% |
| FOLD0 | 0.80 | 1914 | 12.7 / 32.5 / 16.3 / 38.5 | 87.3% |
| FOLD0 | 0.85 | 1914 | 26.5 / 34.4 / 15.9 / 23.1 | 73.5% |
| FOLD0 | 0.90 | 1914 | 52.6 / 29.9 / 9.8 / 7.7 | 47.4% |
| FOLD0 No_ordering=F | 0.80 | 1609 | 12.1 / 33.1 / 16.7 / 38.1 | 87.9% |
| FOLD0 No_ordering=F | 0.85 | 1609 | 26.1 / 35.4 / 15.8 / 22.7 | 73.9% |
| FOLD0 No_ordering=F | 0.90 | 1609 | 52.3 / 29.9 / 10.2 / 7.6 | 47.7% |

## tau별 FT 오답 커버리지

| tau | FT 오답 중 GT-인접 max cos > tau (STAGE 3가 볼 수 있음) |
|---|---|
| 0.80 | 90.6% |
| 0.85 | 79.6% |
| 0.90 | 51.1% |

-> /content/drive/MyDrive/snu ai/snu_pipe

In [9]:
# ---- Phase 1: STAGE 1 그룹 형성 + Phase 2 (b) FT 조대순서 --------------
# STAGE 1/2는 결정적이다. 학습 없음. 세 tau 모두에 대해 만들어 둔다 - 나중에
# epoch x tau 격자로 판정할 때 그대로 쓴다.
fold0_comp, gt_group_order, contiguous_ids, stage2_orders = {}, {}, {}, {}
for tau in EVAL_TAUS:
    comp = {i: components(fold0_cos[i], tau) for i in scored_ids}
    fold0_comp[tau] = comp
    gt_group_order[tau] = {i: group_order_from_rank_label(answer_by_id[i], comp[i])
                           for i in scored_ids}
    # 그룹이 GT에서 연속이 아니면 재조립으로 도달할 수 없다 - 구조 상한.
    contiguous_ids[tau] = [i for i in scored_ids
                           if groups_contiguous(answer_by_id[i], comp[i])]
    stage2_orders[tau] = {
        'b': {i: group_order_from_rank_label(ft_pred[i], comp[i]) for i in scored_ids}
    }
    multi_ids = [i for i in scored_ids if multi_frame_groups(comp[i])]
    print(f'tau={tau:.2f} [학습]  구조상한 '
          f'{len(contiguous_ids[tau]) / len(scored_ids):.4f}  '
          f'multi-frame 샘플 {len(multi_ids)} '
          f'({len(multi_ids) / len(scored_ids) * 100:.1f}%)')


tau=0.80 [학습]  구조상한 0.9357  multi-frame 샘플 1671 (87.3%)
tau=0.85 [학습]  구조상한 0.9206  multi-frame 샘플 1407 (73.5%)
tau=0.90 [학습]  구조상한 0.9253  multi-frame 샘플 908 (47.4%)


In [10]:
# ---- fold-0 프레임 임베딩 로드 (c2-i medoid 선택 입력) -----------------
# c1(캡션 앵커) 팔은 제거했다: fold-0 판정에서 EM 0.08~0.22로 b/c2i(0.49~0.52)에
# 한참 못 미쳤다. CLIP text 유사도가 시간 순서와 무관하다는 게 확인됨.
# 여기서는 c2-i의 그룹 대표(medoid) 선택에 쓸 프레임 임베딩만 로드한다.
import torch.nn.functional as F
from transformers import CLIPModel, CLIPProcessor

feature_blob = torch.load(FOLD0_FEATURES_PT, weights_only=False)
frame_features = {sid: feature_blob['features'][k]
                  for k, sid in enumerate(feature_blob['ids'])}
clip_model_id = feature_blob['model_id']
print(f'CLIP 프레임 임베딩 재사용: {clip_model_id} '
      f'{tuple(feature_blob["features"].shape)}')

def projected(output):
    """transformers<5는 get_*_features가 tensor를, >=5는 BaseModelOutputWithPooling을
    돌려준다. preprocessing/clip_scoring.py의 projected_features와 같은 shim인데,
    그 모듈은 번들에 없으므로 여기 인라인한다. (test 프레임 인코딩에서 사용)"""
    return output if hasattr(output, 'shape') else output.pooler_output


CLIP 프레임 임베딩 재사용: openai/clip-vit-large-patch14 (1915, 4, 768)


In [11]:
# ---- Phase 2 (c2-i): 그룹 대표 재질의 ----------------------------------
# 각 그룹에서 대표 1장씩만 뽑아 FT에 넣는다. near-dup이 제거된 상태라 FT가
# 조대 순서에 집중할 수 있다는 것이 이 팔의 논리다. 그룹이 1개면 자명, 전 프레임이
# 각자 그룹이면 원래 4프레임 질문과 같아지므로 ft_pred를 그대로 쓴다.
from peft import PeftModel
from qwen_vl_utils import process_vision_info
from transformers import AutoModelForImageTextToText, AutoProcessor

processor = AutoProcessor.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME, revision=MODEL_REVISION, dtype=torch.bfloat16, device_map='auto')
ft_model = PeftModel.from_pretrained(base_model, str(WARM_START_ADAPTER)).eval()
processor.tokenizer.padding_side = 'left'

def representative_frame(sample_id, members):
    """그룹 대표 = 멤버 평균 임베딩에 가장 가까운 프레임 (medoid)."""
    frames = F.normalize(frame_features[sample_id].float(), dim=-1)
    subset = frames[[m - 1 for m in members]]
    centroid = F.normalize(subset.mean(0), dim=-1)
    return members[int((subset @ centroid).argmax())]

def frame_path(sample_id, image_number):
    row = train_csv_by_id[sample_id]
    return str(IMAGE_ROOT / sample_id / row[f'Input_{image_number}'])

@torch.no_grad()
def generate_orders(jobs, prompt_fn=build_representative_prompt, batch_size=8):
    """jobs: (key, sentence, [image paths]) -> key -> (local order, parse_ok).
    이 셀은 c2-i(그룹 대표) 전용이라 distinct-moments 프롬프트가 기본값이다."""
    results = {}
    for start in range(0, len(jobs), batch_size):
        chunk = jobs[start:start + batch_size]
        messages = [prompt_fn(sentence, paths) for _, sentence, paths in chunk]
        texts = [processor.apply_chat_template(m, tokenize=False,
                                               add_generation_prompt=True)
                 for m in messages]
        images, _ = process_vision_info(messages)
        inputs = processor(text=texts, images=images, return_tensors='pt',
                           padding=True).to(ft_model.device)
        generated = ft_model.generate(**inputs, max_new_tokens=64, do_sample=False)
        trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated)]
        decoded = processor.batch_decode(trimmed, skip_special_tokens=True)
        for (key, _, paths), text in zip(chunk, decoded):
            results[key] = parse_group_output(text, len(paths))
    return results

stage2_parse_ok = {}
for tau in EVAL_TAUS:
    comp_by_id = fold0_comp[tau]
    c2_jobs, c2_direct = [], {}
    for sample_id in scored_ids:
        comp = comp_by_id[sample_id]
        groups = sorted(group_members(comp))
        if len(groups) <= 1:
            c2_direct[sample_id] = list(groups)
            continue
        if len(groups) == 4:
            # 그룹이 4개면 대표 재질의는 원 질문과 동일 - ft_pred 재사용 (추론 절약)
            c2_direct[sample_id] = stage2_orders[tau]['b'][sample_id]
            continue
        reps = [representative_frame(sample_id, group_members(comp)[g]) for g in groups]
        c2_jobs.append((sample_id, train_csv_by_id[sample_id]['Sentence'],
                        [frame_path(sample_id, r) for r in reps]))
        c2_direct[sample_id] = None

    print(f'(c2-i) tau={tau:.2f} 재질의 대상 {len(c2_jobs)} / 자명·재사용 '
          f'{sum(v is not None for v in c2_direct.values())}')
    c2_raw = generate_orders(c2_jobs)

    # parse 성공 여부를 샘플별로 남긴다. 실패 시 identity fallback이 들어가는데
    # 그게 우연히 맞는 비율이 30%대라, 나중에 정확도를 분리해서 봐야 한다.
    # None = 재질의 자체를 안 한 샘플(그룹 1개는 자명, 4개는 ft_pred 재사용).
    arm_orders, arm_parse_ok = {}, {}
    for sample_id in scored_ids:
        if c2_direct[sample_id] is not None:
            arm_orders[sample_id] = c2_direct[sample_id]
            arm_parse_ok[sample_id] = None
            continue
        groups = sorted(group_members(comp_by_id[sample_id]))
        local_order, ok = c2_raw[sample_id]
        arm_parse_ok[sample_id] = ok
        arm_orders[sample_id] = [groups[pos - 1] for pos in local_order]
    stage2_orders[tau]['c2i'] = arm_orders
    stage2_parse_ok[tau] = arm_parse_ok
    queried = [i for i, v in arm_parse_ok.items() if v is not None]
    failed = [i for i in queried if not arm_parse_ok[i]]
    print(f'  재질의 {len(queried)}, parse 실패 {len(failed)} '
          f'({len(failed) / max(len(queried), 1) * 100:.1f}%)')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

(c2-i) tau=0.80 재질의 대상 934 / 자명·재사용 980
  재질의 934, parse 실패 65 (7.0%)
(c2-i) tau=0.85 재질의 대상 964 / 자명·재사용 950
  재질의 964, parse 실패 63 (6.5%)
(c2-i) tau=0.90 재질의 대상 760 / 자명·재사용 1154
  재질의 760, parse 실패 27 (3.6%)


In [12]:
# ---- Phase 2 비교: 그룹 간 순서 정확도 ---------------------------------
# GT 그룹이 연속인 샘플에서만 정답이 정의된다. 최종 EM은 STAGE 3 학습 후
# Phase 5에서 채우고, 채택은 그 최종 EM 기준이다.
stage2_summary = {}
for tau in EVAL_TAUS:
    def accuracy(orders, ids):
        return sum(orders[i] == gt_group_order[tau][i] for i in ids) / len(ids)

    multi_group_ids = [i for i in contiguous_ids[tau]
                       if len(set(fold0_comp[tau][i])) >= 2]
    rows, disagreement = {}, {}
    for arm in STAGE2_ARMS:
        rows[arm] = {
            'contiguous': accuracy(stage2_orders[tau][arm], contiguous_ids[tau]),
            'multi_group': accuracy(stage2_orders[tau][arm], multi_group_ids),
        }
    disagreement['b_vs_c2i'] = sum(
        stage2_orders[tau]['b'][i] != stage2_orders[tau]['c2i'][i] for i in scored_ids)

    # (c2-i)만 모델 생성이라 parse 실패가 난다. 실패분에는 identity fallback이
    # 들어가고 그 fallback이 30%대 확률로 우연히 맞으므로, 전체 정확도만 보면
    # 모델 실력을 과대평가한다. 실패율·성공분 정확도·fallback 적중률을 함께 남긴다.
    parse_ok = stage2_parse_ok[tau]
    queried = [i for i in contiguous_ids[tau] if parse_ok.get(i) is not None]
    failed = [i for i in queried if not parse_ok[i]]
    parsed = [i for i in queried if parse_ok[i]]
    rows['c2i']['n_queried'] = len(queried)
    rows['c2i']['parse_failure_rate'] = len(failed) / max(len(queried), 1)
    rows['c2i']['contiguous_parsed_only'] = accuracy(
        stage2_orders[tau]['c2i'], parsed) if parsed else None
    rows['c2i']['fallback_lucky_rate'] = (
        sum(stage2_orders[tau]['c2i'][i] == gt_group_order[tau][i] for i in failed)
        / len(failed)) if failed else None

    stage2_summary[tau] = {'accuracy': rows, 'disagreement': disagreement,
                           'n_multi_group': len(multi_group_ids)}

    print(f'tau={tau:.2f}  ' + '  '.join(
        f'{arm}: {rows[arm]["contiguous"]:.4f}/{rows[arm]["multi_group"]:.4f}'
        for arm in STAGE2_ARMS) + '   (연속/그룹>=2)')
    print(f'        c2-i parse 실패 {rows["c2i"]["parse_failure_rate"]:.3f} '
          f'({len(failed)}/{len(queried)})  '
          f'성공분 정확도 {rows["c2i"]["contiguous_parsed_only"]}  '
          f'fallback 우연적중 {rows["c2i"]["fallback_lucky_rate"]}')
    print('        불일치 ' + '  '.join(f'{k} {v}' for k, v in disagreement.items()))

STAGE2_STATE = OUTPUT_DIR / 'stage2_orders.json'
STAGE2_STATE.write_text(json.dumps({
    'train_taus': list(TRAIN_TAUS),
    'eval_taus': list(EVAL_TAUS),
    'summary': {str(t): stage2_summary[t] for t in EVAL_TAUS},
    'orders': {str(t): {arm: stage2_orders[t][arm] for arm in STAGE2_ARMS}
               for t in EVAL_TAUS},
}, ensure_ascii=False, indent=1), encoding='utf-8')
print(f'-> {STAGE2_STATE}')
print('※ 채택은 그룹순서 정확도가 아니라 STAGE 3 통과 후 최종 EM 기준이다.')


tau=0.80  b: 0.9241/0.8710  c2i: 0.9062/0.8406   (연속/그룹>=2)
        c2-i parse 실패 0.048 (39/811)  성공분 정확도 0.8691709844559585  fallback 우연적중 0.46153846153846156
        불일치 b_vs_c2i 146
tau=0.85  b: 0.8496/0.7991  c2i: 0.8275/0.7695   (연속/그룹>=2)
        c2-i parse 실패 0.037 (30/812)  성공분 정확도 0.8005115089514067  fallback 우연적중 0.36666666666666664
        불일치 b_vs_c2i 185
tau=0.90  b: 0.7132/0.6870  c2i: 0.6928/0.6648   (연속/그룹>=2)
        c2-i parse 실패 0.021 (13/617)  성공분 정확도 0.7003311258278145  fallback 우연적중 0.5384615384615384
        불일치 b_vs_c2i 169
-> /content/drive/MyDrive/snu ai/snu_pipelinec_outputs/stage2_orders.json
※ 채택은 그룹순서 정확도가 아니라 STAGE 3 통과 후 최종 EM 기준이다.


In [13]:
# ---- Phase 3: STAGE 3 학습 데이터 (tau마다 별도) -----------------------
# tau마다 그룹이 다르므로 학습 예시도 tau별로 따로 만든다. 각 예시 형식은 동일:
# (그룹 프레임 k장 + 캡션) -> 그룹 내 k-순열. 라틴방격 셔플로 위치 편향 방어.
train_ids = sorted(train_cos)
stage3_examples_by_tau = {}
for tau in TRAIN_TAUS:
    examples = []
    for sample_id in train_ids:
        row = train_csv_by_id[sample_id]
        examples.extend(stage3_examples_for_sample(
            sample_id=sample_id, sentence=row['Sentence'],
            input_files=[row[f'Input_{n}'] for n in range(1, 5)],
            gt_rank_label=answer_by_id[sample_id],
            cosines=train_cos[sample_id], tau=tau, seed=SEED))
    leaked = {e['Id'] for e in examples} & set(holdout_ids)
    assert not leaked, f'STAGE 3 학습셋에 fold-0이 섞였습니다: {sorted(leaked)[:5]}'
    stage3_examples_by_tau[tau] = examples
    group_dist = collections.Counter()
    for key in {(e['Id'], e['group'], e['k']) for e in examples}:
        group_dist[key[2]] += 1
    print(f'tau={tau:.2f}  그룹 k2/3/4={group_dist[2]}/{group_dist[3]}/{group_dist[4]} '
          f'-> 셔플 후 예시 {len(examples)}')

STAGE3_DATA_JSON = OUTPUT_DIR / f'{ARM_NAME}_train_examples.json'
STAGE3_DATA_JSON.write_text(json.dumps(
    {'train_taus': list(TRAIN_TAUS), 'seed': SEED,
     'examples_by_tau': {str(t): len(stage3_examples_by_tau[t]) for t in TRAIN_TAUS}},
    ensure_ascii=False, indent=1), encoding='utf-8')
print(f'-> {STAGE3_DATA_JSON}')


tau=0.80  그룹 k2/3/4=3740/1181/2824 -> 셔플 후 예시 48454
tau=0.85  그룹 k2/3/4=3686/1073/1622 -> 셔플 후 예시 33274
tau=0.90  그룹 k2/3/4=2638/651/557 -> 셔플 후 예시 15866
-> /content/drive/MyDrive/snu ai/snu_pipelinec_outputs/pipelinec_s3v2_train_examples.json


In [14]:
# ---- 데이터셋 / 콜레이터 (qwen3vl_v2_os 노트북 구조 그대로) -------------
from torch.utils.data import DataLoader, Dataset

class GroupOrderDataset(Dataset):
    def __init__(self, examples, image_root, processor):
        self.examples = examples
        self.image_root = Path(image_root)
        self.processor = processor

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        example = self.examples[idx]
        paths = [str(self.image_root / example['Id'] / name)
                 for name in example['images']]
        messages = build_group_prompt(example['sentence'], paths)
        target_text = str(example['target'])
        messages = messages + [{'role': 'assistant',
                                'content': [{'type': 'text', 'text': target_text}]}]

        full_text = self.processor.apply_chat_template(messages, tokenize=False)
        prompt_only = self.processor.apply_chat_template(
            messages[:1], tokenize=False, add_generation_prompt=True)
        image_inputs, _ = process_vision_info(messages)

        full_enc = self.processor(text=[full_text], images=image_inputs,
                                  return_tensors='pt', padding=False)
        prompt_enc = self.processor(text=[prompt_only], images=image_inputs,
                                    return_tensors='pt', padding=False)

        input_ids = full_enc['input_ids'][0]
        labels = input_ids.clone()
        labels[:prompt_enc['input_ids'].shape[1]] = -100

        return {
            'input_ids': input_ids,
            'attention_mask': full_enc['attention_mask'][0],
            'labels': labels,
            'pixel_values': full_enc['pixel_values'],
            'image_grid_thw': full_enc['image_grid_thw'],
            'mm_token_type_ids': full_enc['mm_token_type_ids'][0],
        }


def collate_fn(batch):
    max_len = max(item['input_ids'].shape[0] for item in batch)
    pad_id = processor.tokenizer.pad_token_id

    input_ids, attn_mask, labels, mm_token_type_ids = [], [], [], []
    for item in batch:
        pad_len = max_len - item['input_ids'].shape[0]
        input_ids.append(torch.cat([item['input_ids'],
                                    torch.full((pad_len,), pad_id, dtype=torch.long)]))
        attn_mask.append(torch.cat([item['attention_mask'],
                                    torch.zeros(pad_len, dtype=torch.long)]))
        labels.append(torch.cat([item['labels'],
                                 torch.full((pad_len,), -100, dtype=torch.long)]))
        mm_token_type_ids.append(torch.cat([item['mm_token_type_ids'],
                                            torch.zeros(pad_len, dtype=torch.long)]))

    return {
        'input_ids': torch.stack(input_ids),
        'attention_mask': torch.stack(attn_mask),
        'labels': torch.stack(labels),
        'pixel_values': torch.cat([item['pixel_values'] for item in batch], dim=0),
        'image_grid_thw': torch.cat([item['image_grid_thw'] for item in batch], dim=0),
        'mm_token_type_ids': torch.stack(mm_token_type_ids),
    }


In [15]:
# ---- Phase 4: STAGE 3 학습 (tau마다 qwen3vl_final에서 warm-start) ------
# c2-i에 쓴 추론 전용 모델을 먼저 내려 VRAM을 비운다.
import gc

for name in ('ft_model', 'base_model', 'clip_model'):
    if name in globals():
        del globals()[name]
gc.collect()
torch.cuda.empty_cache()

import bitsandbytes as bnb

model = None   # 학습 루프에서 tau마다 새로 로드한다
processor = AutoProcessor.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)

@torch.no_grad()
def generate_group_orders(jobs, prompt_fn=build_group_prompt, batch_size=8):
    """jobs: (key, sentence, [image paths]) -> key -> (local order, parse_ok).
    전역 model을 쓴다. 기본은 STAGE 3 near-dup 프롬프트; arm b/c2i는 prompt_fn 지정."""
    results = {}
    for start in range(0, len(jobs), batch_size):
        chunk = jobs[start:start + batch_size]
        messages = [prompt_fn(sentence, paths) for _, sentence, paths in chunk]
        texts = [processor.apply_chat_template(m, tokenize=False,
                                               add_generation_prompt=True)
                 for m in messages]
        images, _ = process_vision_info(messages)
        inputs = processor(text=texts, images=images, return_tensors='pt',
                           padding=True).to(model.device)
        generated = model.generate(**inputs, max_new_tokens=64, do_sample=False)
        trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated)]
        for (key, _, paths), text in zip(
                chunk, processor.batch_decode(trimmed, skip_special_tokens=True)):
            results[key] = parse_group_output(text, len(paths))
    return results

def load_trainable(adapter_source):
    """base + adapter를 새로 올려 학습 가능한 warm-start 모델을 만든다."""
    base = AutoModelForImageTextToText.from_pretrained(
        MODEL_NAME, revision=MODEL_REVISION, dtype=torch.bfloat16, device_map='auto')
    m = PeftModel.from_pretrained(base, str(adapter_source), is_trainable=True)
    m.config.use_cache = False
    m.gradient_checkpointing_enable()
    m.enable_input_require_grads()
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    assert trainable == 15_335_424, f'LoRA 형상이 기준 레시피와 다릅니다: {trainable}'
    return m

def build_proxy(tau):
    """early stopping 프록시: 이 tau의 fold-0 multi-frame 그룹 (전체 조립 없음)."""
    rows, jobs = [], []
    for sample_id in scored_ids:
        for group, members in sorted(
                multi_frame_groups(fold0_comp[tau][sample_id]).items()):
            key = (sample_id, group)
            rows.append((sample_id, group, members, key))
            jobs.append((key, train_csv_by_id[sample_id]['Sentence'],
                         [frame_path(sample_id, m) for m in members]))
    return rows, jobs

def proxy_within_accuracy(proxy_rows, proxy_jobs):
    """매 epoch 후 그룹 내 순서 정확도. 학습 모드를 원복하고 돌아온다."""
    model.eval()
    model.gradient_checkpointing_disable()
    model.config.use_cache = True
    processor.tokenizer.padding_side = 'left'
    raw = generate_group_orders(proxy_jobs)
    hit = 0
    for sample_id, group, members, key in proxy_rows:
        local_order, _ = raw[key]
        hit += ([members[position - 1] for position in local_order]
                == within_group_order(answer_by_id[sample_id], members))
    model.config.use_cache = False
    model.gradient_checkpointing_enable()
    model.train()
    return hit / len(proxy_rows)


In [16]:
# ---- tau별 학습 루프 (각 tau 독립 재개/early stop) ---------------------
from torch.utils.data import DataLoader, Subset

def tau_paths(tau):
    prefix = f'{ARM_NAME}_{tau_tag(tau)}'
    return {
        'prefix': prefix,
        'state': OUTPUT_DIR / f'{prefix}_state.jsonl',
        'ckpt': OUTPUT_DIR / f'{prefix}_checkpoint_latest',
        'opt': OUTPUT_DIR / f'{prefix}_optimizer.pt',
        'log': OUTPUT_DIR / f'{prefix}_training_log.json',
        'done': OUTPUT_DIR / f'{prefix}_training_complete.json',
    }

def early_stop_already_reached(hist):
    """proxy 이력만으로 early stop이 이미 걸렸는지 재구성한다(epoch 폴더 유무만
    보면 early stop으로 건너뛴 epoch을 헛되이 다시 학습한다)."""
    best = None
    for h in hist:
        p = h.get('proxy_within_accuracy')
        if p is None:
            continue
        if best is not None and p - best < EARLY_STOP_MIN_DELTA:
            return True
        best = p if best is None else max(best, p)
    return False

def train_one_tau(tau):
    global model
    paths = tau_paths(tau)
    log = json.loads(paths['log'].read_text(encoding='utf-8'))         if paths['log'].is_file() else []
    if (paths['done'].is_file() or len(log) >= MAX_EPOCHS
            or early_stop_already_reached(log)):
        print(f'tau={tau:.2f}: 학습 이미 완료 - 건너뜀')
        return

    dataset = GroupOrderDataset(stage3_examples_by_tau[tau], IMAGE_ROOT, processor)
    proxy_rows, proxy_jobs = build_proxy(tau)

    resume = None
    if paths['state'].is_file():
        lines = [l for l in paths['state'].read_text(encoding='utf-8').splitlines()
                 if l.strip()]
        if lines:
            resume = json.loads(lines[-1])
    if resume is not None and paths['ckpt'].is_dir():
        source = paths['ckpt']
        print(f'tau={tau:.2f}: 재개 epoch {resume["epoch"]} pos {resume["position"]}')
    else:
        source = WARM_START_ADAPTER
        resume = None
        print(f'tau={tau:.2f}: warm-start (qwen3vl_final)')
    print(f'  예시 {len(dataset)}  프록시 그룹 {len(proxy_rows)}  epoch 상한 {MAX_EPOCHS}')

    model = load_trainable(source)
    optimizer = bnb.optim.AdamW8bit(
        [p for p in model.parameters() if p.requires_grad], lr=LEARNING_RATE)
    if resume is not None and paths['opt'].is_file():
        optimizer.load_state_dict(torch.load(paths['opt']))
        print('  optimizer state 복원')
    best_proxy = max((h['proxy_within_accuracy'] for h in log
                      if 'proxy_within_accuracy' in h), default=None)

    def save_adapter(tag):
        path = OUTPUT_DIR / f'{paths["prefix"]}_{tag}'
        assert path.name.startswith(ARM_NAME), f'쓰기 거부: {path}'
        model.save_pretrained(str(path))
        return path

    def write_state(epoch, position):
        with paths['state'].open('a', encoding='utf-8') as handle:
            handle.write(json.dumps({'epoch': epoch, 'position': position,
                                     'at': time.time()}) + chr(10))

    def checkpoint(epoch, position):
        save_adapter('checkpoint_latest')
        torch.save(optimizer.state_dict(), paths['opt'])
        write_state(epoch, position)

    model.train()
    for epoch in range(1, MAX_EPOCHS + 1):
        if (OUTPUT_DIR / f'{paths["prefix"]}_epoch{epoch}').is_dir():
            print(f'  epoch {epoch} 이미 완료 - 건너뜀')
            continue
        position = resume['position'] if (resume and resume['epoch'] == epoch) else 0
        rng = random.Random(SEED * 1000 + int(round(tau * 100)) + epoch)
        order = list(range(len(dataset)))
        rng.shuffle(order)
        loader = DataLoader(Subset(dataset, order[position:]), batch_size=1,
                            shuffle=False, num_workers=2, collate_fn=collate_fn)
        print(f'  epoch {epoch}: {position}번째부터 {len(dataset) - position}개')

        started, running, steps, micro = time.time(), 0.0, 0, 0
        optimizer.zero_grad(set_to_none=True)
        for batch in loader:
            batch = {k: v.to(model.device) for k, v in batch.items()}
            loss = model(**batch).loss
            (loss / GRAD_ACCUM_STEPS).backward()
            running += loss.item()
            steps += 1
            micro += 1
            position += 1
            if micro % GRAD_ACCUM_STEPS == 0:
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], MAX_GRAD_NORM)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                if position % CHECKPOINT_EVERY_STEPS == 0:
                    checkpoint(epoch, position)
                    print(f'    ckpt e{epoch} {position}/{len(dataset)} '
                          f'loss {running / steps:.4f}')

        saved = save_adapter(f'epoch{epoch}')
        proxy = proxy_within_accuracy(proxy_rows, proxy_jobs)
        log.append({'epoch': epoch, 'train_loss': running / max(steps, 1),
                    'wall_seconds': time.time() - started, 'examples_seen': steps,
                    'proxy_within_accuracy': proxy})
        write_state(epoch + 1, 0)
        resume = None
        paths['log'].write_text(json.dumps(log, ensure_ascii=False, indent=1),
                                encoding='utf-8')
        delta = None if best_proxy is None else proxy - best_proxy
        print(f'  epoch {epoch}: loss {log[-1]["train_loss"]:.4f} '
              f'({log[-1]["wall_seconds"] / 3600:.2f}h) proxy {proxy:.4f}'
              + ('' if delta is None else f' ({delta:+.4f})') + f' -> {saved}')
        if delta is not None and delta < EARLY_STOP_MIN_DELTA:
            print(f'  early stop (개선 {delta:+.4f} < {EARLY_STOP_MIN_DELTA})')
            break
        best_proxy = proxy if best_proxy is None else max(best_proxy, proxy)

    saved_epochs = sorted(e for e in range(1, MAX_EPOCHS + 1)
                          if (OUTPUT_DIR / f'{paths["prefix"]}_epoch{e}').is_dir())
    paths['done'].write_text(json.dumps({'epochs': saved_epochs, 'history': log},
                                        ensure_ascii=False, indent=1), encoding='utf-8')
    del optimizer
    globals()['model'] = None
    gc.collect()
    torch.cuda.empty_cache()

EPOCH_ADAPTERS = {}   # {tau: {epoch: dir}}
for tau in TRAIN_TAUS:
    train_one_tau(tau)
    adapters = {e: OUTPUT_DIR / f'{ARM_NAME}_{tau_tag(tau)}_epoch{e}'
                for e in range(1, MAX_EPOCHS + 1)
                if (OUTPUT_DIR / f'{ARM_NAME}_{tau_tag(tau)}_epoch{e}').is_dir()}
    assert adapters, f'tau={tau} epoch 체크포인트가 없습니다'
    EPOCH_ADAPTERS[tau] = adapters

print()
print('tau별 epoch 어댑터:', {t: sorted(EPOCH_ADAPTERS[t]) for t in TRAIN_TAUS})
print('※ 어느 (tau, epoch)을 쓸지는 kill 게이트에서 fold-0 전체 EM으로 정한다.')


tau=0.80: warm-start (qwen3vl_final)
  예시 48454  프록시 그룹 1893  epoch 상한 3


Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

  epoch 1: 0번째부터 48454개


[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


    ckpt e1 2000/48454 loss 0.1257
    ckpt e1 4000/48454 loss 0.1231
    ckpt e1 6000/48454 loss 0.1181
    ckpt e1 8000/48454 loss 0.1160
    ckpt e1 10000/48454 loss 0.1138
    ckpt e1 12000/48454 loss 0.1130
    ckpt e1 14000/48454 loss 0.1123
    ckpt e1 16000/48454 loss 0.1109
    ckpt e1 18000/48454 loss 0.1098
    ckpt e1 20000/48454 loss 0.1074
    ckpt e1 22000/48454 loss 0.1052
    ckpt e1 24000/48454 loss 0.1099
    ckpt e1 26000/48454 loss 0.1246
    ckpt e1 28000/48454 loss 0.1365
    ckpt e1 30000/48454 loss 0.1322
    ckpt e1 32000/48454 loss 0.1283
    ckpt e1 34000/48454 loss 0.1247
    ckpt e1 36000/48454 loss 0.1212
    ckpt e1 38000/48454 loss 0.1176
    ckpt e1 40000/48454 loss 0.1143


KeyboardInterrupt: 

In [ ]:
# ---- 저장 무결성: Drive 재읽기 + shard별 sha256 -------------------------
# EXP-012에서 full-FT 체크포인트 shard 4개가 Drive에서 조용히 사라진 전례가 있다.
manifest = {'arm': ARM_NAME, 'train_taus': list(TRAIN_TAUS), 'dirs': {}}
for adapter_dir in sorted(OUTPUT_DIR.glob(f'{ARM_NAME}_*')):
    if not adapter_dir.is_dir():
        continue
    files = {}
    for file_path in sorted(adapter_dir.rglob('*')):
        if file_path.is_file():
            data = file_path.read_bytes()          # Drive 재읽기 - 잘림 탐지
            files[file_path.name] = {
                'bytes': len(data),
                'sha256': hashlib.sha256(data).hexdigest(),
            }
    manifest['dirs'][adapter_dir.name] = {'file_count': len(files), 'files': files}

# tau x epoch 어댑터는 전부 후보이므로 전부 검증한다.
for tau in TRAIN_TAUS:
    for epoch, adapter_dir in sorted(EPOCH_ADAPTERS[tau].items()):
        weights = adapter_dir / 'adapter_model.safetensors'
        assert weights.is_file() and weights.stat().st_size >= 40 * 1024 * 1024, (
            f'어댑터 가중치가 비정상입니다: {weights}')
        print(f'tau{tau_tag(tau)} epoch{epoch} sha256:',
              manifest['dirs'][adapter_dir.name]['files']
              ['adapter_model.safetensors']['sha256'])

SAVE_MANIFEST = OUTPUT_DIR / f'{ARM_NAME}_save_manifest.json'
SAVE_MANIFEST.write_text(json.dumps(manifest, ensure_ascii=False, indent=1),
                         encoding='utf-8')
print(f'-> {SAVE_MANIFEST}')


In [ ]:
# ---- STAGE 3 fold-0 추론: (tau, epoch) 격자 (tau별 매칭 어댑터) ---------
# 학습이 끝난 뒤 한 번만 돈다. 각 tau의 그룹은 그 tau로 학습된 어댑터로만 푼다
# (외삽 없음). 채택 판정은 다음 셀에서 전체 조립 EM으로만 한다.
base_infer = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME, revision=MODEL_REVISION, dtype=torch.bfloat16, device_map='auto')
model = PeftModel.from_pretrained(base_infer, str(WARM_START_ADAPTER), adapter_name='ft')
model.eval()
model.config.use_cache = True
processor.tokenizer.padding_side = 'left'

def use_adapter(name, path=None):
    if name not in model.peft_config:
        model.load_adapter(str(path), adapter_name=name)
    model.set_adapter(name)

# 각 tau의 fold-0 그룹과 FT 기준선(진단용).
group_rows, ft_within_accuracy = {}, {}
for tau in EVAL_TAUS:
    rows = []
    total, correct = collections.Counter(), collections.Counter()
    for sample_id in scored_ids:
        for group, members in sorted(
                multi_frame_groups(fold0_comp[tau][sample_id]).items()):
            rows.append((sample_id, group, members))
            k = len(members)
            total[k] += 1
            correct[k] += (within_group_order(ft_pred[sample_id], members)
                           == within_group_order(answer_by_id[sample_id], members))
    group_rows[tau] = rows
    ft_within_accuracy[tau] = {k: correct[k] / total[k] for k in total}

stage3_within_grid, within_diag = {}, {}
for tau in EVAL_TAUS:
    jobs = [((sid, g), train_csv_by_id[sid]['Sentence'],
             [frame_path(sid, m) for m in members])
            for sid, g, members in group_rows[tau]]
    for epoch, adapter_dir in sorted(EPOCH_ADAPTERS[tau].items()):
        use_adapter(f'{tau_tag(tau)}_e{epoch}', adapter_dir)
        started = time.time()
        raw = generate_group_orders(jobs)
        within = {}
        total, correct = collections.Counter(), collections.Counter()
        ok_total, ok_correct = collections.Counter(), collections.Counter()
        failed_n = failed_hits = 0
        for sample_id, group, members in group_rows[tau]:
            local_order, ok = raw[(sample_id, group)]
            predicted = [members[position - 1] for position in local_order]
            within[(sample_id, group)] = predicted
            truth = within_group_order(answer_by_id[sample_id], members)
            k = len(members)
            total[k] += 1
            correct[k] += predicted == truth
            if ok:
                ok_total[k] += 1
                ok_correct[k] += predicted == truth
            else:
                failed_n += 1
                failed_hits += predicted == truth
        stage3_within_grid[(tau, epoch)] = within
        within_diag[(tau, epoch)] = {
            'parse_failure_rate': failed_n / len(group_rows[tau]),
            'fallback_lucky_rate': (failed_hits / failed_n) if failed_n else None,
            'by_k': {k: {'n': total[k], 'stage3': correct[k] / total[k],
                         'ft': ft_within_accuracy[tau][k],
                         'stage3_parsed_only': (ok_correct[k] / ok_total[k]
                                                if ok_total[k] else None)}
                     for k in sorted(total)}}
        print(f'tau{tau_tag(tau)} epoch{epoch}: 추론 {time.time() - started:.0f}초')

print()
print('진단(채택 기준 아님): 그룹 내 순서 정확도 STAGE3/FT')
print(f'{"tau":>5} {"epoch":>5} | k=2 S3/FT    k=3 S3/FT    k=4 S3/FT   | parse실패')
for (tau, epoch), data in sorted(within_diag.items()):
    cells = '  '.join(
        (f'{data["by_k"][k]["stage3"]:.3f}/{data["by_k"][k]["ft"]:.3f}'
         if k in data['by_k'] else '     -     ') for k in (2, 3, 4))
    print(f'{tau:5.2f} {epoch:5d} | {cells} | {data["parse_failure_rate"]:9.3f}')

lucky = [d['fallback_lucky_rate'] for d in within_diag.values()
         if d['fallback_lucky_rate'] is not None]
if lucky:
    print()
    print(f'fallback(identity) 우연 적중률 {min(lucky):.3f}~{max(lucky):.3f} '
          f'- parse 실패가 많으면 다음 셀 EM도 그만큼 부풀려진다.')


In [ ]:
# ---- Phase 5 = kill 게이트: 학습 종료 후 1회, 전체 조립 EM으로만 판정 ----
# (tau, epoch, arm) 격자 전부 fold-0 전체 EM을 내고 최고를 0.5298과 비교한다.
import math

def mcnemar_exact_p(b, c):
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    tail = sum(math.comb(n, i) for i in range(k + 1)) / 2.0 ** n
    return min(1.0, 2.0 * tail)

def pipeline_prediction(sample_id, tau, epoch, arm):
    comp = fold0_comp[tau][sample_id]
    within_map = stage3_within_grid[(tau, epoch)]
    within = {}
    for group, members in group_members(comp).items():
        within[group] = (within_map.get((sample_id, group), members)
                         if len(members) >= 2 else members)
    return reassemble(stage2_orders[tau][arm][sample_id], within, comp)

quartile_edges = [0.8468145, 0.895005, 0.929261]
def frame_sim_quartile(sample_id):
    value = max(fold0_cos[sample_id].values())
    return 1 + sum(value > edge for edge in quartile_edges)

grid = {}
for tau in EVAL_TAUS:
    multi = [i for i in scored_ids if multi_frame_groups(fold0_comp[tau][i])]
    for epoch in sorted(EPOCH_ADAPTERS[tau]):
        for arm in STAGE2_ARMS:
            exact = {i: pipeline_prediction(i, tau, epoch, arm) == answer_by_id[i]
                     for i in scored_ids}
            b_only = sum(answer_by_id[i] == ft_pred[i] and not exact[i]
                         for i in scored_ids)
            c_only = sum(exact[i] and answer_by_id[i] != ft_pred[i]
                         for i in scored_ids)
            grid[(tau, epoch, arm)] = {
                'em': sum(exact.values()) / len(scored_ids),
                'em_multi_group': sum(exact[i] for i in multi) / len(multi),
                'n_multi_group': len(multi),
                'mcnemar': {'b_ft_only': b_only, 'c_pipeline_only': c_only,
                            'p': mcnemar_exact_p(b_only, c_only)},
                'exact': exact,
            }

print(f'FT baseline EM = {ft_em:.4f}')
print()
print(f'{"tau":>5} {"epoch":>5} {"arm":>5} {"EM":>8} {"dEM":>8} '
      f'{"grp>=2 EM":>10} {"McNemar p":>10}')
for (tau, epoch, arm), data in sorted(grid.items()):
    print(f'{tau:5.2f} {epoch:5d} {arm:>5} {data["em"]:8.4f} '
          f'{data["em"] - ft_em:+8.4f} {data["em_multi_group"]:10.4f} '
          f'{data["mcnemar"]["p"]:10.4f}')

# tau가 전부 매칭 학습됐으므로 전부 후보다(외삽 없음).
winner = max(grid, key=lambda key: grid[key]['em'])
win_tau, win_epoch, win_arm = winner
best = grid[winner]

# tau별 최고 (epoch, arm) - test.csv를 tau마다 만들 때 쓴다.
per_tau_best = {}
for tau in EVAL_TAUS:
    cand = {k: v for k, v in grid.items() if k[0] == tau}
    key = max(cand, key=lambda k: cand[k]['em'])
    per_tau_best[tau] = {'epoch': key[1], 'arm': key[2], 'em': cand[key]['em']}

by_size, by_quartile = {}, {}
for size in (2, 3, 4):
    ids = [i for i in scored_ids
           if any(len(m) == size
                  for m in multi_frame_groups(fold0_comp[win_tau][i]).values())]
    if ids:
        by_size[size] = {
            'n': len(ids),
            'em': sum(best['exact'][i] for i in ids) / len(ids),
            'ft_em': sum(answer_by_id[i] == ft_pred[i] for i in ids) / len(ids)}
for q in (1, 2, 3, 4):
    ids = [i for i in scored_ids if frame_sim_quartile(i) == q]
    by_quartile[q] = {
        'n': len(ids),
        'em': sum(best['exact'][i] for i in ids) / len(ids),
        'ft_em': sum(answer_by_id[i] == ft_pred[i] for i in ids) / len(ids)}

print()
print('tau별 최고:', {f'{t:.2f}': f'e{v["epoch"]} {v["arm"]} EM {v["em"]:.4f}'
                       for t, v in per_tau_best.items()})
print(f'전체 승자: tau {win_tau}, epoch {win_epoch}, arm {win_arm} '
      f'-> EM {best["em"]:.4f} (baseline {ft_em:.4f})')
print('component 크기별:', {k: f'{v["em"]:.4f} vs FT {v["ft_em"]:.4f}'
                             for k, v in by_size.items()})
print('frame-sim 4분위:', {k: f'{v["em"]:.4f} vs FT {v["ft_em"]:.4f}'
                            for k, v in by_quartile.items()})

accepted = best['em'] > 0.5298
gate_pass = accepted
print()
print(f'kill 게이트 (EM > 0.5298) -> '
      f'{"통과 - STAGE 3 채택" if accepted else "탈락 - STAGE 3 폐기, test는 FT baseline"}')

win_fail_rate = within_diag[(win_tau, win_epoch)]['parse_failure_rate']
if win_fail_rate > PARSE_FAILURE_WARN:
    print(f'경고: 승자 칸 parse 실패율 {win_fail_rate:.3f} > {PARSE_FAILURE_WARN} '
          f'- fallback 우연적중이 EM에 섞임. stage3_parsed_only와 대조할 것.')
print(f'※ 후보 {len(grid)}칸 중 최고를 골랐으므로 다중 비교 편의가 있다. '
      f'승자 EM은 낙관적으로, McNemar p={best["mcnemar"]["p"]:.4f}가 단일 비교 기준.')

judgment = {
    'train_taus': list(TRAIN_TAUS), 'eval_taus': list(EVAL_TAUS),
    'ft_baseline_em': ft_em, 'kill_gate_pass': bool(accepted),
    'within_group_diagnostics': {f'tau{t}_epoch{e}': v
                                 for (t, e), v in within_diag.items()},
    'winner': {'tau': win_tau, 'epoch': win_epoch, 'arm': win_arm},
    'per_tau_best': {str(t): v for t, v in per_tau_best.items()},
    'accepted': bool(accepted), 'n_candidates_compared': len(grid),
    'by_component_size': by_size, 'by_frame_sim_quartile': by_quartile,
    'grid': {f'tau{t}_epoch{e}_{a}': {k: v for k, v in data.items() if k != 'exact'}
             for (t, e, a), data in grid.items()},
}
JUDGMENT_JSON = OUTPUT_DIR / f'{ARM_NAME}_judgment.json'
JUDGMENT_JSON.write_text(json.dumps(judgment, ensure_ascii=False, indent=1),
                         encoding='utf-8')
print(f'-> {JUDGMENT_JSON}')


In [ ]:
# ---- Phase 6: test 예측 (tau마다 전부 생성) ----------------------------
# 각 tau의 최고 (epoch, arm) 구성으로 test.csv를 만든다. tau별로 STAGE 1(그 tau
# 그룹화) + STAGE 2(b/c2i) + STAGE 3(그 tau 어댑터) 전 조립. 통과 시 submission_,
# 탈락 시 reference_rejected_ 접두사. 실제 채택은 전체 승자 tau의 파일이다.
def run_test_predictions():
    from PIL import Image, ImageOps

    test_root = DATA_ROOT / 'test'
    if not test_root.is_dir():
        test_bundle = find_drive_file('snu_colab_test_bundle.tar')
        assert test_bundle is not None, 'snu_colab_test_bundle.tar를 찾지 못했습니다.'
        subprocess.run(['tar', '-xf', str(test_bundle), '-C', str(WORKDIR)], check=True)
    test_rows = read_csv_rows(Path(DATA_DIR) / 'test.csv')
    test_by_id = {r['Id']: r for r in test_rows}
    test_ids = [r['Id'] for r in test_rows]

    def tfp(sample_id, n):
        return str(test_root / sample_id / test_by_id[sample_id][f'Input_{n}'])

    # 1) test 프레임 CLIP 임베딩 -> 6쌍 cosine (tau 무관, 한 번만)
    clip = CLIPModel.from_pretrained(clip_model_id).eval().cuda()
    clip_proc = CLIPProcessor.from_pretrained(clip_model_id)

    @torch.no_grad()
    def encode_images(sample_ids, batch_size=16):
        feats = {}
        for start in range(0, len(sample_ids), batch_size):
            chunk = sample_ids[start:start + batch_size]
            imgs = []
            for sid in chunk:
                for n in range(1, 5):
                    imgs.append(ImageOps.exif_transpose(
                        Image.open(tfp(sid, n))).convert('RGB'))
            inputs = clip_proc(images=imgs, return_tensors='pt').to('cuda')
            vecs = F.normalize(
                projected(clip.get_image_features(**inputs)).float(), dim=-1).cpu()
            for off, sid in enumerate(chunk):
                feats[sid] = vecs[off * 4:(off + 1) * 4]
        return feats

    test_features = encode_images(test_ids)
    del clip
    torch.cuda.empty_cache()

    test_cos = {}
    for sid, vecs in test_features.items():
        sim = vecs @ vecs.T
        test_cos[sid] = {f'cos_{a}{b}': float(sim[a - 1][b - 1])
                         for a, b in ((1, 2), (1, 3), (1, 4),
                                      (2, 3), (2, 4), (3, 4))}
    print(f'test CLIP cosine 완료 ({len(test_ids)}행)')

    # 2) arm b용 4프레임 FT 예측 (원래 FT 프롬프트, tau 무관, 한 번만)
    model.set_adapter('ft')
    full_jobs = [(i, test_by_id[i]['Sentence'], [tfp(i, n) for n in range(1, 5)])
                 for i in test_ids]
    full_raw = generate_group_orders(full_jobs, prompt_fn=build_full_prompt)
    test_ft_rank = {i: temporal_order_to_rank_label(full_raw[i][0]) for i in test_ids}

    prefix = 'submission' if gate_pass else 'reference_rejected'
    written = []
    for tau in EVAL_TAUS:
        epoch = per_tau_best[tau]['epoch']
        comp = {i: components(test_cos[i], tau) for i in test_ids}

        # STAGE 2 (b): 4프레임 예측을 이 tau 경계로 접는다.
        stage2 = {'b': {}, 'c2i': {}}
        for i in test_ids:
            stage2['b'][i] = group_order_from_rank_label(test_ft_rank[i], comp[i])

        # STAGE 2 (c2i): 그룹 대표를 representative 프롬프트로 FT에 재질의.
        model.set_adapter('ft')
        c2_jobs, c2_direct = [], {}
        for i in test_ids:
            members_by_group = group_members(comp[i])
            groups = sorted(members_by_group)
            if len(groups) <= 1 or len(groups) == 4:
                c2_direct[i] = list(groups) if len(groups) <= 1 else stage2['b'][i]
                continue
            frames = test_features[i]
            reps = []
            for g in groups:
                members = members_by_group[g]
                subset = frames[[m - 1 for m in members]]
                centroid = F.normalize(subset.mean(0), dim=-1)
                reps.append(members[int((subset @ centroid).argmax())])
            c2_jobs.append((i, test_by_id[i]['Sentence'], [tfp(i, r) for r in reps]))
            c2_direct[i] = None
        c2_raw = generate_group_orders(c2_jobs, prompt_fn=build_representative_prompt)
        for i in test_ids:
            if c2_direct[i] is not None:
                stage2['c2i'][i] = c2_direct[i]
            else:
                groups = sorted(group_members(comp[i]))
                stage2['c2i'][i] = [groups[p - 1] for p in c2_raw[i][0]]

        # STAGE 3: 이 tau의 최고 epoch 어댑터로 그룹 내 순서.
        use_adapter(f'{tau_tag(tau)}_e{epoch}', EPOCH_ADAPTERS[tau][epoch])
        s3_jobs = [((i, g), test_by_id[i]['Sentence'], [tfp(i, m) for m in members])
                   for i in test_ids
                   for g, members in sorted(multi_frame_groups(comp[i]).items())]
        s3_raw = generate_group_orders(s3_jobs)
        within = {}
        for i in test_ids:
            for g, members in sorted(multi_frame_groups(comp[i]).items()):
                within[(i, g)] = [members[p - 1] for p in s3_raw[(i, g)][0]]

        for arm in STAGE2_ARMS:
            path = OUTPUT_DIR / f'{prefix}_{tau_tag(tau)}_{arm}.csv'
            with open(path, 'w', encoding='utf-8', newline='') as handle:
                writer = csv.writer(handle)
                writer.writerow(['Id', 'Answer'])
                for i in test_ids:
                    w = {g: (within.get((i, g), members) if len(members) >= 2 else members)
                         for g, members in group_members(comp[i]).items()}
                    writer.writerow([i, str(reassemble(stage2[arm][i], w, comp[i]))])
            written.append(path)
            print(f'-> {path}')

    adopted = f'{prefix}_{tau_tag(win_tau)}_{win_arm}.csv'
    md = ['# ' + ('채택 제출' if gate_pass else 'kill 탈락 - 참고용'), '',
          f'- 전체 승자: tau {win_tau}, epoch {win_epoch}, arm {win_arm}, '
          f'fold-0 EM {best["em"]:.4f} (baseline {ft_em:.4f})']
    if gate_pass:
        md.append(f'- 실제 제출 = `{adopted}`')
    else:
        md.append('- **실제 제출 = FT baseline(qwen3vl_final).** '
                  'reference_rejected_*.csv는 참고용, 제출하지 말 것.')
    md.append('- tau별 파일 전부 보관. test 점수로 고르지 않았다(fold-0 EM 기준).')
    (OUTPUT_DIR / 'SUBMISSION_CHOICE.md').write_text(
        chr(10).join(md) + chr(10), encoding='utf-8')
    print()
    print(('채택: ' + adopted) if gate_pass
          else '탈락: 실제 제출은 FT baseline. tau별 파일은 참고용.')

run_test_predictions()


In [ ]:
# ---- Phase 7: 산출물 회수 ----------------------------------------------
import shutil

from google.colab import files

bundle_dir = Path('/content/pipelinec_artifacts')
if bundle_dir.exists():
    shutil.rmtree(bundle_dir)
bundle_dir.mkdir(parents=True)

for pattern in ('*.md', '*.json', 'submission_*.csv', 'reference_rejected_*.csv'):
    for path in OUTPUT_DIR.glob(pattern):
        shutil.copy2(path, bundle_dir / path.name)
# 채택 epoch의 어댑터만 담는다(전부 담으면 zip이 180MB가 된다).
# 나머지 epoch 체크포인트는 Drive에 그대로 남아 있다.
adopted_adapter = EPOCH_ADAPTERS.get(win_epoch)
if adopted_adapter and adopted_adapter.is_dir():
    shutil.copytree(adopted_adapter, bundle_dir / adopted_adapter.name)

archive = shutil.make_archive('/content/pipelinec_artifacts', 'zip', str(bundle_dir))
print(f'{archive} ({Path(archive).stat().st_size / 1024 ** 2:.1f} MiB)')
for name in sorted(p.name for p in bundle_dir.iterdir()):
    print(' ', name)
files.download(archive)
